# Decision Tree From Scratch

**What you'll build:** A binary decision tree classifier using Gini impurity for splits.

**Prerequisites:** Recursion, numpy, basic classification concepts.

## Concept Recap

**Gini impurity:** $G = 1 - \sum_k p_k^2$. 0 = pure node (all one class), max at 0.5 (50/50 split).

**Information gain:** $IG = G_{parent} - \frac{n_L}{n}G_L - \frac{n_R}{n}G_R$

**Algorithm:**
1. For each feature and threshold, compute information gain
2. Split on the (feature, threshold) pair with highest gain
3. Recurse on left and right subsets
4. Stop when: max depth reached, or node is pure, or too few samples

**Prediction:** traverse tree from root to leaf; return majority class at leaf.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # leaf class

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow(X, y, depth=0)

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1 - np.sum(p ** 2)

    def _best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, None, None
        g_parent = self._gini(y)
        n = len(y)
        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left = y[X[:, feat] <= thresh]
                right = y[X[:, feat] > thresh]
                if len(left) == 0 or len(right) == 0:
                    continue
                gain = g_parent - (len(left)/n)*self._gini(left) - (len(right)/n)*self._gini(right)
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh

    def _grow(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(value=np.bincount(y).argmax())
        feat, thresh = self._best_split(X, y)
        if feat is None:
            return Node(value=np.bincount(y).argmax())
        mask = X[:, feat] <= thresh
        left = self._grow(X[mask], y[mask], depth + 1)
        right = self._grow(X[~mask], y[~mask], depth + 1)
        return Node(feature=feat, threshold=thresh, left=left, right=right)

    def _traverse(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
tree = DecisionTree(max_depth=4)
tree.fit(X_tr, y_tr)
print(f"From-scratch accuracy: {np.mean(tree.predict(X_te) == y_te):.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

clf = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
print(f"sklearn accuracy: {clf.score(X_te, y_te):.4f}")

plt.figure(figsize=(14, 6))
plot_tree(clf, feature_names=load_iris().feature_names, class_names=load_iris().target_names,
          filled=True, rounded=True, fontsize=8)
plt.title('sklearn Decision Tree (depth=4)')
plt.tight_layout()
plt.show()

## Exercises

1. **Entropy splitting:** Replace Gini with Shannon entropy $H = -\sum p_k \log_2 p_k$ and compare accuracy.
2. **Regression tree:** Modify to predict mean(y) at leaf nodes, using MSE as split criterion.
3. **Feature importance:** Count how many times each feature is used as a split and at what depth. Normalize to get importance scores.

## Summary

- Trees split greedily on the feature/threshold maximizing information gain
- Gini and entropy give nearly identical results in practice
- Deeper trees overfit — control with max_depth and min_samples_split

**Next:** [Random Forest](random-forest.ipynb)